# Speech-to-Speech Translation for Low-Resource Languages: A Luganda Case Study (Colab-safe Edition)

## 1. Learning outcomes

By the end of this notebook, you should be able to:
1. Explain what **speech-to-speech translation** is.
2. Build a **cascade S2ST pipeline** for Luganda.
3. Run a more advanced **multilingual direct speech model**.
4. Evaluate outputs using **WER**, **CER**, and **BLEU**.
5. Discuss the challenges of **low-resource speech translation** for African languages.


## 2. Visual intuition

### Cascade pipeline

```text
Luganda Speech  ──ASR──>  Luganda Text  ──MT──>  English Text  ──TTS──>  English Speech
```

### End-to-end / unified multilingual model

```text
Luganda Speech  ────────────────>  English Text or English Speech
```

The cascade approach is easier to teach and debug. Unified multilingual models are elegant, but heavier and more experimental in classroom settings.


In [ ]:
from IPython.display import Markdown, display

diagram = "Luganda Speech
      |
      v
  [ ASR ]  ---> Luganda Text
      |
      v
 [  MT  ]  ---> English Text
      |
      v
 [ TTS  ]  ---> English Speech"

display(Markdown(f"```text
{diagram}
```"))


## 3. Install dependencies

This notebook uses a mix of practical and research-oriented tools:
- `transformers` for Whisper, NLLB, SeamlessM4T
- `datasets` for dataset access
- `librosa` and `soundfile` for audio handling
- `gTTS` for a lightweight English speech demo
- `jiwer` and `sacrebleu` for evaluation

> In Colab, run the next cell once. If prompted, restart the runtime after installation.


In [ ]:
!pip -q install -U transformers datasets accelerate sentencepiece librosa soundfile ipywidgets jiwer sacrebleu gTTS evaluate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

## 4. Imports and helper utilities


In [ ]:
import os
import io
import json
import base64
import numpy as np
import soundfile as sf
import librosa
import torch

from IPython.display import Audio, display, Markdown, Javascript
from google.colab import files, output


## 5. Upload **or record** a Luganda audio sample

You now have two ways to provide audio:

### Option A — Upload an existing file
Run the next cell and upload a `.wav`, `.mp3`, `.m4a`, or similar audio file.

### Option B — Record directly in Colab
Run the recorder cell after that, speak a short Luganda sentence, then stop the recording.

Whichever option you use last will become the active `audio_path`.


In [ ]:
# Option A: Upload an existing audio file
uploaded = files.upload()

if len(uploaded) > 0:
    audio_path = list(uploaded.keys())[0]
    print("Using uploaded audio file:", audio_path)
else:
    audio_path = None
    print("No file uploaded yet. You can record audio in the next cell.")


Saving 2-bd_95_5-2022-11-01_T11.00.02 2.wav to 2-bd_95_5-2022-11-01_T11.00.02 2 (1).wav
Using uploaded audio file: 2-bd_95_5-2022-11-01_T11.00.02 2 (1).wav


### Option B — Record Luganda audio directly in Colab

Run the next cell, allow microphone access, speak, then click stop. The cell will save the recording to `recorded_luganda.wav` and set `audio_path` automatically.


In [ ]:
def record_audio_colab(filename="recorded_luganda.wav", sample_rate=48000):
    js = f'''
    async function recordAudio() {{
      const div = document.createElement("div");
      const startBtn = document.createElement("button");
      const stopBtn = document.createElement("button");
      const status = document.createElement("p");

      startBtn.textContent = "Start recording";
      stopBtn.textContent = "Stop recording";
      stopBtn.disabled = true;
      status.textContent = "Click start and speak in Luganda.";

      div.appendChild(startBtn);
      div.appendChild(stopBtn);
      div.appendChild(status);
      document.body.appendChild(div);

      const stream = await navigator.mediaDevices.getUserMedia({{audio: true}});
      let recorder;
      let chunks = [];

      const waitForStop = new Promise((resolve) => {{
        startBtn.onclick = () => {{
          chunks = [];
          recorder = new MediaRecorder(stream);
          recorder.ondataavailable = e => chunks.push(e.data);
          recorder.onstop = async () => {{
            const blob = new Blob(chunks, {{type: "audio/webm"}});
            const reader = new FileReader();
            reader.readAsDataURL(blob);
            reader.onloadend = () => resolve(reader.result);
          }};
          recorder.start();
          startBtn.disabled = true;
          stopBtn.disabled = false;
          status.textContent = "Recording...";
        }};

        stopBtn.onclick = () => {{
          recorder.stop();
          startBtn.disabled = false;
          stopBtn.disabled = true;
          status.textContent = "Saving recording...";
        }};
      }});

      return await waitForStop;
    }}
    recordAudio();
    '''
    audio_data = output.eval_js(js)
    header, encoded = audio_data.split(",", 1)
    with open("recorded_luganda.webm", "wb") as f:
        f.write(base64.b64decode(encoded))

    audio, sr = librosa.load("recorded_luganda.webm", sr=16000, mono=True)
    sf.write(filename, audio, 16000)
    return filename

# Uncomment the next two lines when you want to record:
# audio_path = record_audio_colab()
# print("Using recorded audio file:", audio_path)


## 6. Normalize audio to 16 kHz mono

Most speech models expect mono audio around **16 kHz**.


In [ ]:
def load_audio_16k(path):
    audio, sr = librosa.load(path, sr=16000, mono=True)
    return audio.astype(np.float32), sr

def save_wav(path, audio, sr=16000):
    sf.write(path, audio, sr)

if audio_path is None:
    normalized_path = None
    print("Please upload or record audio first.")
else:
    wave, sr = load_audio_16k(audio_path)
    normalized_path = "luganda_input_16k.wav"
    save_wav(normalized_path, wave, sr)

    print("Saved normalized audio to:", normalized_path)
    print("Sampling rate:", sr)
    print("Duration (seconds):", round(len(wave) / sr, 2))
    display(Audio(normalized_path))


Saved normalized audio to: luganda_input_16k.wav
Sampling rate: 16000
Duration (seconds): 9.98


## 7. Route A — Practical cascade pipeline

This is the best teaching route because each stage is visible and debuggable.

### Step A1. ASR: Luganda speech → Luganda text

We offer two ASR options:

1. **Fast baseline:** OpenAI Whisper
2. **Uganda-focused option:** Sunbird AI SALT Whisper fine-tune

Start with Whisper for reliability. Then try the Sunbird model if you want a more region-specific experiment.



### Step A1. Choose an ASR checkpoint

This notebook now uses an **open Whisper model by default** so it runs in Colab without gated-access errors.

You have two options:

**Option 1 — Default (recommended for class demos)**
- `openai/whisper-small`
- open access
- easier to run immediately

**Option 2 — Optional Uganda-focused model**
- `Sunbird/asr-whisper-large-v2-salt`
- requires:
  1. a Hugging Face account,
  2. access approval on the model page,
  3. login in Colab with a read token.

If you do **not** already have access to the Sunbird model, keep `USE_SUNBIRD_ASR = False`.


In [ ]:

# Optional: authenticate to Hugging Face only if you want to use the gated Sunbird model.
# Leave RUN_HF_LOGIN = False for the default open Whisper model.

RUN_HF_LOGIN = True

if RUN_HF_LOGIN:
    from huggingface_hub import login
    login()  # paste your Hugging Face read token when prompted
else:
    print("Skipping Hugging Face login. Open Whisper model will work without this step.")


In [ ]:

from transformers import pipeline

# Set this to True only if:
# 1) you have requested/received access to the Sunbird model on Hugging Face, and
# 2) you have logged in successfully in the previous cell.
USE_SUNBIRD_ASR = False

asr_model_name = "Sunbird/asr-whisper-large-v2-salt" if USE_SUNBIRD_ASR else "openai/whisper-small"
print("Loading ASR model:", asr_model_name)

asr = pipeline(
    "automatic-speech-recognition",
    model=asr_model_name,
    device=0 if torch.cuda.is_available() else -1,
)


Loading ASR model: openai/whisper-small


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

In [ ]:
# Robust ASR helper:
# Instead of passing only a file path to the pipeline, we load the waveform
# ourselves and pass {"array": ..., "sampling_rate": ...}. This avoids
# KeyError issues such as 'num_frames' in some Colab/audio backend setups.

def run_asr(audio_path, task="transcribe"):
    if audio_path is None:
        raise ValueError("audio_path is None. Upload or record audio first.")

    audio_array, sr = librosa.load(audio_path, sr=16000, mono=True)
    audio_input = {
        "array": audio_array.astype(np.float32),
        "sampling_rate": sr
    }

    try:
        result = asr(audio_input, generate_kwargs={"task": task})
    except TypeError:
        # Some checkpoints/pipeline versions may not accept generate_kwargs cleanly.
        result = asr(audio_input)

    return result

# Run ASR
if normalized_path is not None:
    asr_result = run_asr(normalized_path, task="transcribe")
    luganda_text = asr_result["text"]
    print("Recognized Luganda text:")
    print(luganda_text)
else:
    luganda_text = None
    print("Upload or record audio first.")


KeyError: 'num_frames'

**Why this ASR helper matters:** some Colab and backend combinations raise `KeyError: 'num_frames'` when a raw file path is passed directly into the ASR pipeline. Loading the waveform first with `librosa` and passing `{"array": ..., "sampling_rate": ...}` is usually more reliable.



### ASR troubleshooting: gated model vs audio-format errors

There are two common failure modes:

1. **`OSError: You are trying to access a gated repo`**  
   This means you selected the Sunbird checkpoint without completing Hugging Face access + login.  
   Fix:
   - set `USE_SUNBIRD_ASR = False`, **or**
   - request access on the model page and run the login cell above.

2. **`KeyError: 'num_frames'`**  
   This usually comes from passing a raw file path directly to the ASR pipeline.  
   In this notebook we avoid that by loading the waveform with `librosa` first and then passing:
   ```python
   {"array": audio_array, "sampling_rate": 16000}
   ```


### Step A1b. Fine-tune a Whisper model for Luganda ASR

After students see zero-shot ASR, it is useful to show how to **adapt Whisper to Luganda** using supervised speech-text pairs.

This section is intentionally designed for **Colab teaching**:
- use a **small checkpoint** (`openai/whisper-small`) instead of a very large model
- use a **small subset** first so students can run the pipeline end to end
- show the full workflow: **load data → preprocess → train → evaluate → test inference**

**Recommended teaching message:** fine-tuning matters because zero-shot ASR is convenient, but a domain- or language-adapted model often performs much better on local accents, vocabulary, names, and code-switching.

**References for this section**
- Hugging Face blog: *Fine-Tune Whisper For Multilingual ASR with Transformers*  
  https://huggingface.co/blog/fine-tune-whisper
- Hugging Face Audio Course: *Fine-tuning the ASR model*  
  https://huggingface.co/learn/audio-course/chapter5/fine-tuning
- Whisper model documentation  
  https://huggingface.co/docs/transformers/model_doc/whisper

> Practical note: this classroom section teaches the workflow. On free Colab, use a **small subset** for demonstration. For serious Luganda training, increase the data size and training time.

In [ ]:
# Optional extra install for the fine-tuning section
!pip -q install -U evaluate

#### A1b-1. Load a Luganda speech dataset

A convenient starting point is **Common Voice**. In the Hugging Face ecosystem, Luganda is typically referenced with the language code **`lg`**.

For a fast classroom demo, we take only a **small subset** first. Later, you can expand these sample sizes or replace Common Voice with your own Luganda dataset.

Expected columns:
- `audio`: waveform + sampling rate
- `sentence`: ground-truth transcript

In [ ]:
from datasets import load_dataset, DatasetDict, Audio
import evaluate

MAX_TRAIN_SAMPLES = 100
MAX_EVAL_SAMPLES = 20

common_voice = DatasetDict()
common_voice["train"] = load_dataset(
    "mozilla-foundation/common_voice_17_0",
    "lg",
    split="train+validation"
)
common_voice["test"] = load_dataset(
    "mozilla-foundation/common_voice_17_0",
    "lg",
    split="test"
)

# Keep the tutorial small enough for Colab demonstration
if MAX_TRAIN_SAMPLES is not None:
    common_voice["train"] = common_voice["train"].shuffle(seed=42).select(
        range(min(MAX_TRAIN_SAMPLES, len(common_voice["train"])))
    )

if MAX_EVAL_SAMPLES is not None:
    common_voice["test"] = common_voice["test"].shuffle(seed=42).select(
        range(min(MAX_EVAL_SAMPLES, len(common_voice["test"])))
    )

# Cast audio to 16 kHz for Whisper
common_voice = common_voice.cast_column("audio", Audio(sampling_rate=16000))

print(common_voice)
print("Train samples:", len(common_voice["train"]))
print("Test samples:", len(common_voice["test"]))

#### A1b-2. Load the Whisper processor and model

We start from a pre-trained Whisper checkpoint and adapt it to Luganda transcripts.

For a broad, portable classroom setup, this example avoids forcing a language token during training. That makes the notebook easier to adapt when working with custom or under-resourced languages.

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

finetune_base_model = "openai/whisper-small"
print("Fine-tuning base model:", finetune_base_model)

whisper_processor = WhisperProcessor.from_pretrained(finetune_base_model)
whisper_model = WhisperForConditionalGeneration.from_pretrained(finetune_base_model)

# Important for custom ASR fine-tuning workflows:
# we do not force decoder prompts here
whisper_model.config.forced_decoder_ids = None
whisper_model.config.suppress_tokens = []

#### A1b-3. Prepare the dataset

Whisper needs:
- **input features** from the audio waveform
- **labels** from the transcript text

In [ ]:
def prepare_whisper_batch(batch):
    audio = batch["audio"]

    batch["input_features"] = whisper_processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = whisper_processor.tokenizer(batch["sentence"]).input_ids
    return batch

common_voice_prepared = common_voice.map(
    prepare_whisper_batch,
    remove_columns=common_voice["train"].column_names,
    num_proc=1
)

common_voice_prepared

#### A1b-4. Data collator and evaluation metric

We use **WER (Word Error Rate)**, which is standard in ASR:
- lower is better
- `0.0` means a perfect transcript

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

wer_metric = evaluate.load("wer")

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], np.ndarray]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Remove BOS token if it is already present
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=whisper_processor)

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = whisper_processor.tokenizer.pad_token_id

    pred_str = whisper_processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = whisper_processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

#### A1b-5. Training configuration

These settings are deliberately light for **teaching on Colab**.  
For better results later:
- increase the training data
- train for more steps or epochs
- try gradient accumulation, mixed precision, or LoRA/PEFT

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

finetuned_model_dir = "whisper-small-luganda-demo"

training_args = Seq2SeqTrainingArguments(
    output_dir=finetuned_model_dir,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=10,
    max_steps=50,
    evaluation_strategy="steps",
    eval_steps=10,
    save_steps=10,
    logging_steps=5,
    predict_with_generate=True,
    generation_max_length=128,
    fp16=torch.cuda.is_available(),
    report_to=[],
    load_best_model_at_end=False,
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=whisper_model,
    train_dataset=common_voice_prepared["train"],
    eval_dataset=common_voice_prepared["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=whisper_processor.feature_extractor,
)

trainer

#### A1b-6. Train and evaluate

Run the next cell to perform a **small demo fine-tuning job**.

On free Colab, this is still compute-heavy, so it is normal if training takes time. For a lecture, you can:
1. run only a very small training job live, or  
2. pre-train earlier and use the saved checkpoint during class.

In [ ]:
train_result = trainer.train()
eval_results = trainer.evaluate()

print("Evaluation results:")
print(eval_results)

#### A1b-7. Test the fine-tuned Luganda model on one sample

This lets students compare:
- **zero-shot Whisper**
- **fine-tuned Whisper**
- and optionally a Uganda-focused external checkpoint

In [ ]:
sample = common_voice["test"][0]
sample_audio = sample["audio"]["array"]
sample_sr = sample["audio"]["sampling_rate"]
sample_ref = sample["sentence"]

pred_ids = whisper_model.generate(
    inputs=whisper_processor.feature_extractor(
        sample_audio,
        sampling_rate=sample_sr,
        return_tensors="pt"
    ).input_features.to(whisper_model.device),
    max_new_tokens=128
)

sample_pred = whisper_processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)[0]

print("Reference:")
print(sample_ref)
print("\nPrediction:")
print(sample_pred)

#### A1b-8. Teaching notes: what students should notice

1. **Data size matters.** With only a tiny subset, this demo teaches the workflow, not the best possible accuracy.
2. **Local vocabulary matters.** Fine-tuning helps with names, places, and pronunciation patterns that zero-shot models often miss.
3. **Evaluation matters.** Always compare predictions to references with WER, and inspect qualitative errors manually.
4. **Low-resource reality.** For Luganda, collecting more high-quality audio-text pairs can be just as important as changing the model.

### Step A2. Translate Luganda text → English text with NLLB-200

We use the **distilled 600M parameter** checkpoint because it is much lighter for Colab than the larger variants.

Luganda language code: **`lug_Latn`**
English language code: **`eng_Latn`**


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

mt_model_name = "facebook/nllb-200-distilled-600M"
print("Loading MT model:", mt_model_name)

tokenizer = AutoTokenizer.from_pretrained(mt_model_name, src_lang="lug_Latn")
mt_model = AutoModelForSeq2SeqLM.from_pretrained(mt_model_name)


In [ ]:
def translate_lug_to_en(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    generated_tokens = mt_model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"),
        max_new_tokens=256
    )
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

english_text = translate_lug_to_en(luganda_text)
print("English translation:")
print(english_text)


### Step A3. TTS: English text → English speech

Because target speech support for Luganda is still limited in mainstream multilingual speech stacks, we use **English speech output** for a clean demo.


In [ ]:
from gtts import gTTS

output_tts_path = "luganda_to_english_output.mp3"
gtts = gTTS(text=english_text, lang="en")
gtts.save(output_tts_path)

print("Saved TTS audio to:", output_tts_path)
display(Audio(output_tts_path))


### Step A4. Wrap the full cascade into one function


In [ ]:
def cascade_luganda_s2st(audio_path):
    if audio_path is None:
        raise ValueError("audio_path is None. Upload or record audio first.")

    asr_out = run_asr(audio_path, task="transcribe")
    source_text = asr_out["text"]
    target_text = translate_lug_to_en(source_text)

    tts_path = "cascade_output.mp3"
    gTTS(text=target_text, lang="en").save(tts_path)

    return {
        "source_text_lug": source_text,
        "target_text_en": target_text,
        "tts_path": tts_path,
    }

if normalized_path is not None:
    full_result = cascade_luganda_s2st(normalized_path)
    print(json.dumps(full_result, indent=2, ensure_ascii=False))
    display(Audio(full_result["tts_path"]))
else:
    print("Upload or record audio first.")


## 8. Route B — Advanced multilingual speech model with SeamlessM4T v2

This section demonstrates a **more unified multilingual approach**.

### Important note
API details for large speech models can evolve across `transformers` releases, so treat this section as an **advanced lab**.

You can try:
- official model: `facebook/seamless-m4t-v2-large`
- lighter experimentation: `facebook/hf-seamless-m4t-medium`
- Uganda-focused research model: `Geneline-X/seamless-m4t-v2-sunbird-multilingual-v1`


In [ ]:
from transformers import AutoProcessor, SeamlessM4Tv2Model

USE_ADVANCED_ROUTE = False
advanced_model_name = "facebook/seamless-m4t-v2-large"

if USE_ADVANCED_ROUTE:
    print("Loading advanced model:", advanced_model_name)
    processor = AutoProcessor.from_pretrained(advanced_model_name)
    seamless_model = SeamlessM4Tv2Model.from_pretrained(advanced_model_name)
else:
    print("Set USE_ADVANCED_ROUTE = True to run this section.")


In [ ]:
# Speech-to-text translation (Luganda speech -> English text)
if USE_ADVANCED_ROUTE:
    audio_array, sr = librosa.load(normalized_path, sr=16000, mono=True)
    inputs = processor(audios=audio_array, sampling_rate=sr, return_tensors="pt")

    text_tokens = seamless_model.generate(
        **inputs,
        tgt_lang="eng",
        generate_speech=False,
    )
    advanced_english_text = processor.decode(text_tokens[0].tolist(), skip_special_tokens=True)
    print("Advanced route English text:")
    print(advanced_english_text)


In [ ]:
# Speech-to-speech translation (Luganda speech -> English speech)
# Depending on package version and checkpoint support, the exact return type may vary.
if USE_ADVANCED_ROUTE:
    audio_array, sr = librosa.load(normalized_path, sr=16000, mono=True)
    inputs = processor(audios=audio_array, sampling_rate=sr, return_tensors="pt")

    audio_out = seamless_model.generate(
        **inputs,
        tgt_lang="eng",
        generate_speech=True,
    )

    if isinstance(audio_out, tuple):
        speech = audio_out[0]
    else:
        speech = audio_out

    speech_np = speech[0].detach().cpu().numpy().squeeze()
    sf.write("seamless_output.wav", speech_np, 16000)
    display(Audio("seamless_output.wav"))


## 9. Optional dataset exploration

This is useful when teaching why Luganda is a **low-resource but improving** language setting.


In [ ]:
from datasets import load_dataset

SHOW_DATASET_EXAMPLE = False

if SHOW_DATASET_EXAMPLE:
    cv_lg = load_dataset("mozilla-foundation/common_voice_17_0", "lg", split="train", streaming=True)
    first_item = next(iter(cv_lg))
    print(first_item.keys())
    print("Sentence:", first_item.get("sentence"))
else:
    print("Set SHOW_DATASET_EXAMPLE = True to stream a Common Voice Luganda example.")


## 10. Evaluation: WER, CER, and BLEU

This section lets students compare the system output against a small reference.

### What to enter
- `reference_luganda`: the correct Luganda transcription for the spoken sample
- `reference_english`: the correct English translation for the same sample

### Metrics
- **WER (Word Error Rate)**: lower is better
- **CER (Character Error Rate)**: lower is better
- **BLEU**: higher is better

For classroom demos, start with one short sentence and inspect the kinds of errors the model makes.


In [ ]:
from jiwer import wer, cer
from sacrebleu import sentence_bleu

# Replace these with the correct references for YOUR audio sample.
reference_luganda = "ssebo oli otya leero"
reference_english = "sir how are you today"

# Model hypotheses
hyp_luganda = (luganda_text or "").lower().strip()
hyp_english = (english_text or "").lower().strip()

print("Reference Luganda :", reference_luganda)
print("Hypothesis Luganda:", hyp_luganda)
print()
print("Reference English :", reference_english)
print("Hypothesis English:", hyp_english)
print()

if hyp_luganda:
    print("WER :", wer(reference_luganda, hyp_luganda))
    print("CER :", cer(reference_luganda, hyp_luganda))
else:
    print("WER/CER skipped because Luganda hypothesis is empty.")

if hyp_english:
    bleu = sentence_bleu(hyp_english, [reference_english]).score
    print("BLEU:", bleu)
else:
    print("BLEU skipped because English hypothesis is empty.")


### Optional: reusable evaluation helper

This helper makes it easy to evaluate multiple examples during a live tutorial.


In [ ]:
def evaluate_example(reference_lug, hypothesis_lug, reference_en, hypothesis_en):
    results = {}

    if hypothesis_lug and reference_lug:
        results["wer"] = wer(reference_lug.lower().strip(), hypothesis_lug.lower().strip())
        results["cer"] = cer(reference_lug.lower().strip(), hypothesis_lug.lower().strip())
    else:
        results["wer"] = None
        results["cer"] = None

    if hypothesis_en and reference_en:
        results["bleu"] = sentence_bleu(
            hypothesis_en.lower().strip(),
            [reference_en.lower().strip()]
        ).score
    else:
        results["bleu"] = None

    return results

results = evaluate_example(reference_luganda, hyp_luganda, reference_english, hyp_english)
print(results)


## 11. Error analysis prompts for students

Ask students to inspect the pipeline and answer:

1. Did the ASR confuse **Luganda** with **English** or another language?
2. Did the MT system mistranslate names, greetings, or code-switching?
3. Did TTS sound natural for the translated sentence?
4. Where did the biggest error likely occur: **ASR**, **MT**, or **TTS**?
5. How would results change with:
   - cleaner audio,
   - a domain-adapted ASR model,
   - a Luganda-specific TTS model,
   - a direct speech-to-speech model?


## 12. Low-resource challenges in Luganda speech translation

### Common challenges
- Limited parallel speech-to-text and speech-to-speech data
- Code-switching between **Luganda** and **English**
- Accent and dialect variability
- Proper names and local entities
- Sparse high-quality TTS resources
- Domain mismatch: studio audio vs phone audio vs field audio

### Practical research responses
- Fine-tune multilingual models on local datasets
- Use self-supervised pretraining (e.g. wav2vec 2.0, HuBERT)
- Add noisy and phone-speech augmentation
- Build small domain-specific evaluation sets
- Use community data programs such as Common Voice


## 13. Suggested classroom flow

### Bachelor level
- Explain the cascade pipeline
- Upload a short clip
- Show ASR → MT → TTS outputs

### Master's level
- Compare Whisper baseline vs Uganda-focused ASR
- Compute WER and BLEU
- Discuss where errors come from

### PhD level
- Compare cascade vs unified multilingual models
- Discuss source-target coverage limits
- Propose research ideas for direct Luganda S2ST and Luganda TTS


## 14. Suggested exercises

1. Replace the audio sample with a different Luganda speaker.
2. Compare `openai/whisper-small` against a Uganda-focused ASR checkpoint.
3. Change the translation target from English to another NLLB-supported language.
4. Create a 10-sentence mini evaluation set and compute average WER/BLEU.
5. Test code-switched speech (Luganda + English) and analyze failures.


## 15. Research extension ideas

- Build **Luganda ASR** from Common Voice + local corpora
- Build **Luganda TTS** from curated female/male voice datasets
- Explore **unit-based speech translation** for low-resource settings
- Fine-tune **SeamlessM4T v2** or related multilingual models for Ugandan languages
- Study how code-switching affects ASR and MT
- Build evaluation benchmarks for **health**, **education**, or **agriculture** speech domains

Future and Ongoing research:

- End to End Speech to Speech system with no need of MT & TTS Model
- Direct S2ST models, such as Translatotron 2 and UnitY, retain speaker
identity, reduce latency, and improve translation naturalness by preserving vocal
characteristics and prosody (intonation patterns in speech)
- limited by data sparsity, high
computational costs, and generalization challenges for low-resource languages

## 16. Key citations for your tutorial

### Core S2ST / multilingual speech papers
- Vaswani et al. (2017). **Attention Is All You Need.** NeurIPS. https://arxiv.org/abs/1706.03762
- Jia et al. (2019). **Direct Speech-to-Speech Translation with a Sequence-to-Sequence Model.** https://arxiv.org/abs/1904.06037
- Jia et al. (2021). **Translatotron 2.** https://arxiv.org/abs/2107.08661
- SeamlessM4T publication page. https://ai.meta.com/research/publications/seamlessm4t-massively-multilingual-multimodal-machine-translation/

### Speech representation learning
- Baevski et al. (2020). **wav2vec 2.0.** https://arxiv.org/abs/2006.11477
- Hsu et al. (2021). **HuBERT.** https://arxiv.org/abs/2106.07447

### Translation / low-resource multilingual MT
- Costa-jussà et al. (2022). **No Language Left Behind.** https://ai.meta.com/blog/nllb-200-high-quality-machine-translation/

### Datasets and tools relevant to Luganda
- Mozilla Common Voice Luganda datasheet — https://datacollective.mozillafoundation.org/datasets/cmn2hjqe001n0mm07lbfcq6bp
- FLEURS dataset — https://huggingface.co/datasets/google/fleurs
- Sunbird AI SALT ASR model card — https://huggingface.co/Sunbird/asr-whisper-large-v2-salt
- Optional Luganda-English parallel corpus — https://huggingface.co/datasets/kambale/luganda-english-parallel-corpus


## 17. Final takeaways

- For a **reliable demo today**, the best classroom pipeline is:
  **Luganda speech → Luganda text → English text → English speech**
- For a **research-forward demo**, add **SeamlessM4T v2** as an advanced multilingual path.
- The most important insight is not only the model, but also the **data quality**, **domain fit**, and **evaluation design**.

This makes Luganda an excellent case study for **low-resource speech translation**.


**Colab note:** for the most reliable classroom demo, keep the default open Whisper checkpoint and treat the Sunbird model as an optional extension once access has been granted.